# 🐱 내 고양이 찾기 CNN — 메인 노트북
## Day 4 필수 기술 전체 포함: Conv→Pool×2 + Dense + Dropout + Callbacks

**포함 기술:**
- reshape 4D 텐서 변환 `reshape(-1, 64, 64, 1)`
- Conv2D + MaxPooling2D (×2 블록)
- Dense + Dropout
- `model.summary()` — output shape, 파라미터 수 확인
- 파라미터 수 직접 계산 + weight sharing 효율성
- Padding·Stride 출력 크기 공식
- `jit_compile=False`
- ModelCheckpoint + EarlyStopping

---
## 1. 이론 정리

### 1-1. 출력 Feature Map 크기 공식

$$\text{output size} = \frac{W - F + 2P}{S} + 1$$

| 기호 | 의미 | 예시 |
|------|------|------|
| W | 입력 크기 | 64 |
| F | 필터(커널) 크기 | 3 |
| P | 패딩 | 1 (same) / 0 (valid) |
| S | 스트라이드 | 1 |

**예시: 64×64 입력, Conv2D(32, 3, padding='same', strides=1)**
→ (64 - 3 + 2×1) / 1 + 1 = **64** (same 패딩이라 크기 유지)

**예시: padding='valid'**
→ (64 - 3 + 0) / 1 + 1 = **62**

### 1-2. 파라미터 수 계산

**Conv2D(32, kernel_size=3, input_channels=1)**
- 가중치: 3 × 3 × 1 × 32 = 288
- 바이어스: 32
- **총: 288 + 32 = 320개**

### 1-3. Weight Sharing의 효율성

만약 64×64×1 입력을 Dense로 처리하면?
- Dense(32): 64 × 64 × 1 × 32 + 32 = **131,104개** 파라미터
- Conv2D(32, 3): **320개** 파라미터
- → Conv2D가 약 **400배** 효율적!

이유: 하나의 필터가 이미지 전체를 슬라이딩하며 같은 가중치를 공유 (weight sharing)

### 1-4. MaxPooling2D

- (2,2) 풀링: 2×2 영역에서 최대값만 선택 → 공간 크기 **절반**으로 축소
- 효과: 연산량 감소 + **위치 불변성** (고양이가 살짝 이동해도 같은 특징 감지)

---
## 2. 데이터 준비

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
from sklearn.model_selection import train_test_split

# --- 설정 ---
IMG_SIZE = 64  # 64×64로 리사이즈
DATA_DIR = Path("data")

In [ ]:
def load_images(folder: Path, label: int, img_size: int = IMG_SIZE):
    """폴더에서 이미지를 읽어 numpy 배열로 반환"""
    images = []
    valid_ext = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

    for img_path in sorted(folder.iterdir()):
        if img_path.suffix.lower() not in valid_ext:
            continue
        try:
            img = Image.open(img_path).convert("L")  # 흑백 변환 (1채널)
            img = img.resize((img_size, img_size))
            images.append(np.array(img))
        except Exception as e:
            print(f"  ⚠️ {img_path.name} 로드 실패: {e}")

    labels = np.full(len(images), label)
    return np.array(images), labels


# 학습 데이터 로드
print("📂 학습 데이터 로드 중...")
my_train, my_train_y = load_images(DATA_DIR / "train" / "my_cat", label=1)
other_train, other_train_y = load_images(DATA_DIR / "train" / "other_cats", label=0)

# 검증 데이터 로드
print("📂 검증 데이터 로드 중...")
my_val, my_val_y = load_images(DATA_DIR / "val" / "my_cat", label=1)
other_val, other_val_y = load_images(DATA_DIR / "val" / "other_cats", label=0)

# 합치기
train_images = np.concatenate([my_train, other_train])
train_labels = np.concatenate([my_train_y, other_train_y])
val_images = np.concatenate([my_val, other_val])
val_labels = np.concatenate([my_val_y, other_val_y])

print(f"\n학습: {len(train_images)}장 (내 고양이 {len(my_train)} + 다른 고양이 {len(other_train)})")
print(f"검증: {len(val_images)}장 (내 고양이 {len(my_val)} + 다른 고양이 {len(other_val)})")

### 2-1. 정규화 + 4D 텐서 변환 (reshape)

**왜 reshape이 필요한가?**
- 원본: `(N, 64, 64)` — 3D 텐서 (배치, 높이, 너비)
- Conv2D 입력: `(N, 64, 64, 1)` — 4D 텐서 (배치, 높이, 너비, **채널**)
- 흑백이라 채널=1이지만 Conv2D는 반드시 채널 차원이 필요

Fashion MNIST에서 `reshape(-1, 28, 28, 1)` 하는 것과 동일한 원리!

In [ ]:
# 0~255 → 0~1 정규화
train_images = train_images.astype("float32") / 255.0
val_images = val_images.astype("float32") / 255.0

# ★ 핵심: 4D 텐서로 reshape (채널 차원 추가)
# Fashion MNIST: reshape(-1, 28, 28, 1)
# 우리 프로젝트: reshape(-1, 64, 64, 1)
train_images = train_images.reshape(-1, IMG_SIZE, IMG_SIZE, 1)
val_images = val_images.reshape(-1, IMG_SIZE, IMG_SIZE, 1)

print(f"학습 데이터 shape: {train_images.shape}")  # (N, 64, 64, 1)
print(f"검증 데이터 shape: {val_images.shape}")
print(f"레이블 클래스: 0=다른고양이, 1=내고양이")

In [ ]:
# 샘플 이미지 확인
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle("샘플 이미지 (위: 내 고양이 / 아래: 다른 고양이)", fontsize=13)

for i, ax in enumerate(axes[0]):
    idx = np.where(train_labels == 1)[0]
    if i < len(idx):
        ax.imshow(train_images[idx[i]].squeeze(), cmap="gray")
    ax.axis("off")

for i, ax in enumerate(axes[1]):
    idx = np.where(train_labels == 0)[0]
    if i < len(idx):
        ax.imshow(train_images[idx[i]].squeeze(), cmap="gray")
    ax.axis("off")

plt.tight_layout()
plt.show()

---
## 3. CNN 모델 구성
### Conv→Pool × 2 + Flatten + Dense + Dropout (교재 08-2 구조)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras import layers

model = keras.Sequential([
    # ===== 첫 번째 Conv→Pool 블록 =====
    layers.Conv2D(32, kernel_size=3, activation="relu",
                  padding="same", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.MaxPooling2D(pool_size=2),

    # ===== 두 번째 Conv→Pool 블록 =====
    layers.Conv2D(64, kernel_size=3, activation="relu", padding="same"),
    layers.MaxPooling2D(pool_size=2),

    # ===== 분류기 =====
    layers.Flatten(),
    layers.Dense(100, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),  # 이진 분류
])

### 3-1. model.summary() — output shape & 파라미터 수 확인

In [ ]:
model.summary()

### 3-2. 파라미터 수 직접 계산 & 검증

**Layer 1: Conv2D(32, 3, input_channels=1)**
- 가중치: 3 × 3 × 1 × 32 = 288
- 바이어스: 32
- **총: 320** ← summary에서 확인!

**Layer 2: MaxPooling2D(2)**
- 파라미터: **0** (학습 파라미터 없음, 단순 연산)

**Layer 3: Conv2D(64, 3, input_channels=32)**
- 가중치: 3 × 3 × 32 × 64 = 18,432
- 바이어스: 64
- **총: 18,496**

**Layer 4: MaxPooling2D(2)**
- 파라미터: **0**

**Flatten 후 Dense(100)**
- 입력: 16 × 16 × 64 = 16,384
- 가중치: 16,384 × 100 = 1,638,400
- 바이어스: 100
- **총: 1,638,500**

### 3-3. Feature Map 크기 추적 (Padding·Stride 공식 검증)

| 레이어 | 계산 | 출력 크기 |
|--------|------|-----------|
| Input | — | 64 × 64 × 1 |
| Conv2D(32, 3, same) | (64-3+2)/1+1 = 64 | 64 × 64 × 32 |
| MaxPool(2) | 64 / 2 = 32 | **32 × 32 × 32** |
| Conv2D(64, 3, same) | (32-3+2)/1+1 = 32 | 32 × 32 × 64 |
| MaxPool(2) | 32 / 2 = 16 | **16 × 16 × 64** |
| Flatten | 16×16×64 | **16,384** |

In [ ]:
# 코드로 검증
print("=== 파라미터 수 검증 ===")
for layer in model.layers:
    name = layer.name
    params = layer.count_params()
    if hasattr(layer, "output_shape"):
        print(f"{name:25s} | output: {str(layer.output_shape):20s} | params: {params:,}")

---
## 4. 컴파일 + 콜백 설정

In [ ]:
# ★ jit_compile=False: Apple Silicon(M1/M2) 또는 특정 GPU에서
#    XLA 컴파일 오류가 날 수 있어 False로 설정
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
    jit_compile=False,  # ← XLA 비활성화
)

### jit_compile=False가 필요한 이유

- `jit_compile=True`: XLA(Accelerated Linear Algebra) 컴파일러 사용
- Apple Silicon, 일부 GPU, WSL 환경에서 XLA 관련 오류 발생 가능
- 안정성을 위해 명시적으로 `jit_compile=False` 설정
- 성능 차이는 미미 (대규모 모델이 아닌 이상)

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping
from pathlib import Path

Path("models").mkdir(exist_ok=True)

# ★ ModelCheckpoint: 검증 정확도가 가장 높은 모델을 자동 저장
checkpoint = ModelCheckpoint(
    filepath="models/best_cat_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1,
)

# ★ EarlyStopping: val_loss가 5 에포크 동안 개선 안 되면 조기 종료
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True,
    verbose=1,
)

---
## 5. 모델 학습

In [ ]:
history = model.fit(
    train_images, train_labels,
    epochs=30,
    batch_size=16,
    validation_data=(val_images, val_labels),
    callbacks=[checkpoint, early_stop],
    verbose=1,
)

---
## 6. 결과 시각화

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 정확도
ax1.plot(history.history["accuracy"], label="학습 정확도")
ax1.plot(history.history["val_accuracy"], label="검증 정확도")
ax1.set_title("Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 손실
ax2.plot(history.history["loss"], label="학습 손실")
ax2.plot(history.history["val_loss"], label="검증 손실")
ax2.set_title("Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 최종 성능 출력
best_val_acc = max(history.history["val_accuracy"])
print(f"\n🏆 최고 검증 정확도: {best_val_acc:.4f} ({best_val_acc:.2%})")

---
## 7. 예측 테스트

In [ ]:
def predict_cat(image_path: str, model):
    """새 이미지로 예측"""
    img = Image.open(image_path).convert("L")  # 흑백
    img = img.resize((IMG_SIZE, IMG_SIZE))
    img_array = np.array(img).astype("float32") / 255.0
    img_array = img_array.reshape(1, IMG_SIZE, IMG_SIZE, 1)  # 4D 텐서

    prob = model.predict(img_array, verbose=0)[0][0]

    plt.figure(figsize=(4, 4))
    plt.imshow(img, cmap="gray")
    plt.axis("off")

    if prob > 0.5:
        plt.title(f"🐱 내 고양이! ({prob:.1%})", fontsize=14)
    else:
        plt.title(f"😿 다른 고양이 ({1-prob:.1%})", fontsize=14)
    plt.show()


# 사용 예시 (test_images 폴더에 사진 넣고 실행):
# predict_cat("test_images/test1.jpg", model)

---
## 8. Dense만 쓴 모델과 비교 (Weight Sharing 효율성)

In [ ]:
# Dense-only 모델 (비교용)
dense_model = keras.Sequential([
    layers.Flatten(input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    layers.Dense(100, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(1, activation="sigmoid"),
])

print("=" * 50)
print("CNN 모델 파라미터 수:")
print(f"  총: {model.count_params():,}개")
print()
print("Dense-only 모델 파라미터 수:")
print(f"  총: {dense_model.count_params():,}개")
print()
print(f"→ Dense 모델이 {dense_model.count_params() / model.count_params():.1f}배 더 많은 파라미터 사용")
print(f"→ CNN의 weight sharing 덕분에 적은 파라미터로도 공간적 특징을 효과적으로 학습!")